# Faseeh: AI analysis server + evaluation

Run the cells from top to bottom. **Runtime → Change runtime type → T4 GPU** first.

1. Install the model and tools
2. Start the analysis server (downloads the Quran Muaalem model, about 2 GB, the first time)
3. Open a public link to it and get the address for the website
4. Run the evaluation (numbers and charts for the report)

Model: [Quran Muaalem](https://github.com/obadx/quran-muaalem) (paper: arXiv 2509.00094), QML 1.0 licence.

In [ ]:
!nvidia-smi -L || echo 'No GPU: Runtime > Change runtime type > T4 GPU'

In [ ]:
!pip install -q "quran-muaalem[engine]" levenshtein soundfile datasets matplotlib
# the model code needs this transformers version (newer ones removed a name it imports)
!pip install -q "transformers==4.55.2"
!ffmpeg -version | head -1

In [ ]:
%%writefile faseeh_server.py
"""
Faseeh analysis server: the real AIEngine behind the web page.

POST /analyze  (body = the recorded audio, header X-Ayahs = "112:1,112:2")
  1. converts the audio to 16 kHz mono,
  2. runs the Quran Muaalem model (obadx/muaalem-model-v3_2) to get the phonemes the reader actually said,
  3. builds the correct Hafs phonemes for the selected ayahs with quran-transcript,
  4. aligns the two and returns every mistake with its ayah, word, letter and type.

Response (same shape the page already understands):
  { "112:1": [ {"positionIndex": 3, "letter": "ح", "errorType": "إبدال الحاء هاءً",
               "kind": "normal", "expected": "حَ", "heard": "هَ", "tajweed": {"ar": "...", "en": "..."}} ],
    "_meta": { "predicted": "...", "reference": "...", "seconds": 4.2 } }

Model and phonetic script: Quran Muaalem (https://github.com/obadx/quran-muaalem), QML 1.0 license.
The page must show the licence's AI advisory notice to every user (it does).
"""
import json
import os
import subprocess
import threading
from types import SimpleNamespace
import tempfile
import time
import urllib.request

import Levenshtein as lv
import numpy as np
import soundfile as sf
import torch
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from starlette.concurrency import run_in_threadpool

from quran_transcript import Aya, MoshafAttributes, explain_error, quran_phonetizer
from quran_transcript.phonetics.sifa import chunck_phonemes
from quran_muaalem import Muaalem

SR = 16000
MAX_SECONDS = float(os.environ.get("FASEEH_MAX_SECONDS", "330"))   # a whole surah (An-Naba is about 4.5 minutes)
MAX_BYTES = 25 * 1024 * 1024
CHUNK_SECONDS = 20          # long recordings are analysed in pieces cut at pauses
MISMATCH = 0.45             # above this share of differences the reader did not recite the selected ayahs
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Hafs (via ash-Shatibiyyah) as recited by Mishary Alafasy, the reciter the page plays
MOSHAF = MoshafAttributes(
    rewaya="hafs",
    recitation_speed="murattal",
    madd_monfasel_len=4,
    madd_mottasel_len=4,
    madd_mottasel_waqf=4,
    madd_aared_len=4,
)

print(f"Loading Quran Muaalem on {DEVICE} ...")
MODEL = Muaalem(device=DEVICE, dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32)
print("Model ready.")

app = FastAPI(title="Faseeh analysis server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

ARABIC_LETTERS = set("ءابتثجحخدذرزسشصضطظعغفقكلمنهوي" "أإآؤئىةٱ")

# Letter swaps the page has a dedicated explanation for (expected letter, heard letter) -> rules.json key
SWAPS = {
    ("ح", "ه"): "إبدال الحاء هاءً",
    ("ص", "س"): "إبدال الصاد سينًا",
    ("ث", "س"): "إبدال الثاء سينًا",
    ("ع", "ء"): "إبدال العين همزة",
    ("ط", "ت"): "ترقيق الطاء",
    ("ق", "ك"): "ترقيق القاف",
}
# Heavy letters: heard light (sifat level) -> rules.json key
LIGHT = {"ق": "ترقيق القاف", "ط": "ترقيق الطاء", "خ": "ترقيق الخاء"}


def to_wave(raw: bytes) -> np.ndarray:
    """Any browser recording (webm/ogg/mp4/wav) -> 16 kHz mono float32, using ffmpeg."""
    with tempfile.TemporaryDirectory() as d:
        src, dst = os.path.join(d, "in"), os.path.join(d, "out.wav")
        with open(src, "wb") as f:
            f.write(raw)
        r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-ac", "1", "-ar", str(SR), dst],
                           capture_output=True)
        if r.returncode != 0:
            raise HTTPException(400, "Could not read the audio: " + r.stderr.decode(errors="ignore")[:200])
        wave, _ = sf.read(dst, dtype="float32")
    return wave


# Half a second of silence on each side: speech models tend to drop the very first and last sound of a clip
# when it starts or ends abruptly (e.g. the first hamzah of an ayah).
PAD = np.zeros(int(0.5 * SR), dtype=np.float32)


def padded(wave: np.ndarray) -> np.ndarray:
    return np.concatenate([PAD, wave.astype(np.float32), PAD])


def keep(err) -> bool:
    """False for differences that are not mistakes in Hafs:
    - the madd at a stop (aared) and the leen madd may be 2, 4 or 6 counts;
    - the separated / connected madd may be 4 or 5 counts, and a one-count difference on a long madd is within
      how reciters and the model measure length.
    The natural madd (2 counts) must stay exact."""
    if err.error_type != "tajweed" or err.expected_len is None or err.predicted_len is None:
        return True
    names = " ".join(r.name.en for r in (err.ref_tajweed_rules or []))
    if "Aared" in names or "Leen" in names:
        return err.predicted_len < 2
    if "Monfasel" in names or "Mottasel" in names or "Lazem" in names:
        return abs(err.expected_len - err.predicted_len) > 1
    return True


def letter_at(text: str, start: int, end: int) -> str:
    """First real letter inside text[start:end], else the nearest letter before it."""
    for i in range(start, max(end, start + 1)):
        if i < len(text) and text[i] in ARABIC_LETTERS:
            return text[i]
    for i in range(min(start, len(text) - 1), -1, -1):
        if text[i] in ARABIC_LETTERS:
            return text[i]
    return ""


def uthmani_index_of_phoneme(mappings, ph_idx: int) -> int:
    for i, m in enumerate(mappings):
        if m is not None and not m.deleted and m.pos[0] <= ph_idx < m.pos[1]:
            return i
    return 0


def error_key(err) -> str:
    """Our explanation key (rules.json) for one ReciterError."""
    if err.error_type == "tajweed":
        names = " ".join(type(r).__name__ for r in (err.ref_tajweed_rules or []) + (err.missing_tajweed_rules or []))
        if "Madd" in names:
            return "خطأ في المد"
        if "Ghonna" in names or "Idgham" in names:
            return "ترك الغنة"
        if "Qalqala" in names:
            return "ترك القلقلة"
        return "خطأ في التجويد"
    if err.error_type == "tashkeel":
        exp, got = err.expected_ph or "", err.preditected_ph or ""
        if exp[:1] in ("ن", "م") and exp[:1] == got[:1] and len(exp) != len(got):
            return "ترك الغنة"                    # the nasal letter was held too short: it is the ghunnah
        return "خطأ في الحركة"
    if err.speech_error_type == "replace":
        exp, got = err.expected_ph[:1], (err.preditected_ph or "")[:1]
        return SWAPS.get((exp, got), "إبدال حرف")
    if err.speech_error_type == "delete":
        return "حذف حرف"
    return "زيادة حرف"


def tajweed_names(err):
    rules = (err.ref_tajweed_rules or []) + (err.missing_tajweed_rules or [])
    return {"ar": "، ".join(r.name.ar for r in rules), "en": ", ".join(r.name.en for r in rules)} if rules else None


def ref_to_pred(ref: str, pred: str) -> list[int]:
    """For every position in the reference phoneme string, the matching position in the predicted one."""
    m = [0] * (len(ref) + 1)
    for tag, i1, i2, j1, j2 in lv.opcodes(ref, pred):
        for k in range(i1, i2 + 1):
            m[k] = j1 if tag == "delete" else min(j2, j1 + (k - i1))
    m[len(ref)] = len(pred)
    return m


def split_predicted(ref_parts: list[str], predicted: str) -> tuple[list[str], list[int]]:
    """Cut the predicted phoneme string at the same places as the per-ayah references.
    Returns the pieces and where each piece starts in the predicted string."""
    m = ref_to_pred("".join(ref_parts), predicted)
    pieces, starts, prev, total = [], [], 0, 0
    for p in ref_parts[:-1]:
        total += len(p)
        at = m[total]
        pieces.append(predicted[prev:at]); starts.append(prev)
        prev = at
    pieces.append(predicted[prev:]); starts.append(prev)
    return pieces, starts


def sifat_errors(ref_out, pred_sifat, pred_text):
    """Heavy letters read light and missing qalqalah, from the model's sifat predictions (equal phoneme groups only)."""
    ref_groups = chunck_phonemes(ref_out.phonemes)
    pred_groups = chunck_phonemes(pred_text)
    if len(pred_groups) != len(pred_sifat) or len(ref_groups) != len(ref_out.sifat):
        return []
    found, ref_pos = [], [0]
    for g in ref_groups:
        ref_pos.append(ref_pos[-1] + len(g))
    for tag, i1, i2, j1, j2 in lv.opcodes([g[0] for g in ref_groups], [g[0] for g in pred_groups]):
        if tag != "equal":
            continue
        for a, b in zip(range(i1, i2), range(j1, j2)):
            ref_s, got = ref_out.sifat[a], pred_sifat[b]
            letter = ref_groups[a][0]
            got_tafkheem = getattr(getattr(got, "tafkheem_or_taqeeq", None), "text", None)
            got_qalqla = getattr(getattr(got, "qalqla", None), "text", None)
            if letter in LIGHT and ref_s.tafkheem_or_taqeeq == "mofakham" and got_tafkheem == "moraqaq":
                found.append((ref_pos[a], letter, LIGHT[letter]))
            elif ref_s.qalqla == "moqalqal" and got_qalqla == "not_moqalqal":
                found.append((ref_pos[a], letter, "ترك القلقلة"))
    return found


@app.get("/health")
def health():
    return {"ok": True, "device": DEVICE}


# The reciter's surah recordings, served from here so the website can decode them exactly (the audio site does
# not let other pages read its files). Downloaded once per surah and kept on disk; the very same files the
# word timings in data/reciters.json were made for.
AUDIO_URL = "https://download.quranicaudio.com/qdc/mishari_al_afasy/murattal/{}.mp3"
AUDIO_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "audio_cache")
AUDIO_LOCK = threading.Lock()


@app.get("/audio/{surah}.mp3")
def reciter_audio(surah: int):
    if not 1 <= surah <= 114:
        raise HTTPException(404, "No such surah.")
    path = os.path.join(AUDIO_DIR, f"{surah}.mp3")
    with AUDIO_LOCK:
        if not os.path.exists(path):
            os.makedirs(AUDIO_DIR, exist_ok=True)
            try:
                req = urllib.request.Request(AUDIO_URL.format(surah), headers={"User-Agent": "Faseeh"})
                data = urllib.request.urlopen(req, timeout=90).read()
            except Exception as e:
                raise HTTPException(502, f"Could not fetch the recording: {e}")
            with open(path + ".part", "wb") as f:
                f.write(data)
            os.replace(path + ".part", path)
    return FileResponse(path, media_type="audio/mpeg", headers={"Cache-Control": "public, max-age=604800"})


@app.post("/analyze")
async def analyze(request: Request):
    keys = [k for k in request.headers.get("x-ayahs", "").split(",") if ":" in k]
    if not keys or len(keys) > 60:
        raise HTTPException(400, "Header X-Ayahs is required, e.g. 112:1,112:2")
    try:
        ayahs = [(int(s), int(a)) for s, a in (k.split(":") for k in keys)]
        for s, a in ayahs:
            Aya(sura_idx=s, aya_idx=a)
    except Exception:
        raise HTTPException(400, "Unknown ayah in X-Ayahs.")
    body = await request.body()
    if len(body) > MAX_BYTES:
        raise HTTPException(413, "The recording is too large.")
    wave = await run_in_threadpool(to_wave, body)
    seconds = len(wave) / SR
    if seconds < 0.5:
        raise HTTPException(400, "The recording is empty or too short.")
    if seconds > MAX_SECONDS:
        raise HTTPException(413, f"Please record at most {MAX_SECONDS:.0f} seconds at a time.")

    if float(np.sqrt(np.mean(wave ** 2))) < 1e-3:
        raise HTTPException(422, "silence")             # nothing was recorded (microphone muted?)
    return await run_in_threadpool(analyze_wave, wave, ayahs)


# ---------------------------------------------------------------------------------------------------
# Running the model: one request at a time on the GPU, long recordings in pieces cut at the quietest point.
MODEL_LOCK = threading.Lock()


def pieces(wave: np.ndarray) -> list[np.ndarray]:
    longest = int(CHUNK_SECONDS * SR)
    if len(wave) <= longest:
        return [wave]
    frame = int(0.03 * SR)
    energy = np.convolve(wave.astype(np.float64) ** 2, np.ones(frame) / frame, mode="same")
    out, start = [], 0
    while len(wave) - start > longest:
        lo, hi = start + longest // 2, start + longest
        cut = lo + int(np.argmin(energy[lo:hi]))          # a pause between words or ayahs
        out.append(wave[start:cut]); start = cut
    out.append(wave[start:])
    return out


def run_model(wave: np.ndarray, whole) -> SimpleNamespace:
    """Model output for the whole recording: phoneme text, per-phoneme confidence and sifat, pieces joined."""
    texts, probs, sifat = [], [], []
    for piece in pieces(wave):
        with MODEL_LOCK:
            out = MODEL([padded(piece)], [whole], sampling_rate=SR)[0]
        texts.append(out.phonemes.text)
        probs.extend(float(p) for p in out.phonemes.probs)
        sifat.extend(out.sifat)
    return SimpleNamespace(phonemes=SimpleNamespace(text="".join(texts), probs=probs), sifat=sifat)


# ---------------------------------------------------------------------------------------------------
# Avoiding false alarms. A correct reading must not be marked wrong, so a mistake is only reported when
#  1. the model hears it in at least VOTES of the 3 versions of the recording (normal speed, 5% slower,
#     5% faster): a real mistake is heard in all of them, a slip of the model usually in one only, and
#  2. for a replaced or added sound, the model is at least MIN_CONF sure of the sound it heard.
# The values are chosen with evaluate.py (sweep), which measures both false alarms and missed mistakes.
#  3. the same mistake is not also "heard" when expert reciters read that ayah correctly (calibration.json,
#     made by calibrate.py from reciters that are NOT used for testing): those spots are the model's own
#     habitual slips, not the reader's.
# config.json (written by `evaluate.py sweep` on separate tuning data) overrides the defaults below.
VOTES = int(os.environ.get("FASEEH_VOTES", "2"))
MIN_CONF = float(os.environ.get("FASEEH_MIN_CONF", "0.5"))
USE_CALIBRATION = True
CALIB_MODE = "exact"          # "exact": same word and same mistake type; "word": any mistake on that word
SKIP_KINDS = set()            # mistake kinds not reported at all (e.g. {"tashkeel"} if they are not reliable)
SPEEDS = (1.0, 0.95, 1.05)
HERE = os.path.dirname(os.path.abspath(__file__))
try:
    _cfg = json.load(open(os.path.join(HERE, "config.json")))
    VOTES, MIN_CONF, USE_CALIBRATION = int(_cfg["votes"]), float(_cfg["min_conf"]), bool(_cfg["calibration"])
    CALIB_MODE, SKIP_KINDS = _cfg.get("calib_mode", "exact"), set(_cfg.get("skip_kinds", []))
    print("config.json:", _cfg)
except FileNotFoundError:
    pass
try:
    CALIBRATION = {k: {(w, t) for w, t in v} for k, v in json.load(open(os.path.join(HERE, "calibration.json"))).items()}
    print("calibration.json:", sum(len(v) for v in CALIBRATION.values()), "habitual slips in", len(CALIBRATION), "ayahs")
except FileNotFoundError:
    CALIBRATION = {}


def stretch(wave: np.ndarray, factor: float) -> np.ndarray:
    """The recording played `factor` times as fast (simple resampling)."""
    if factor == 1.0:
        return wave
    n = max(1, int(len(wave) / factor))
    return np.interp(np.linspace(0, len(wave) - 1, n), np.arange(len(wave)), wave).astype(np.float32)


def raw_mistakes(wave: np.ndarray, ayahs: list[tuple[int, int]]):
    """Every difference the model hears, separately for each version of the recording, with its confidence."""
    texts = [Aya(sura_idx=s, aya_idx=a).get().uthmani for s, a in ayahs]
    refs = [quran_phonetizer(t, MOSHAF, remove_spaces=True, sura_idx=s) for t, (s, _) in zip(texts, ayahs)]
    whole = quran_phonetizer(" ".join(texts), MOSHAF, remove_spaces=True, sura_idx=ayahs[0][0])
    t0 = time.time()
    outs = [run_model(stretch(wave, f), whole) for f in SPEEDS]
    infer_s = time.time() - t0

    variants = []
    for v, out in enumerate(outs):
        predicted = out.phonemes.text
        probs = [float(p) for p in out.phonemes.probs]
        if len(probs) != len(predicted):
            probs = None                                   # tokens are not single characters: no confidence
        parts, starts = split_predicted([r.phonemes for r in refs], predicted)
        per_ayah = {}
        for (s, a), text, ref, heard, off in zip(ayahs, texts, refs, parts, starts):
            items = {}
            m = ref_to_pred(ref.phonemes, heard)
            for err in explain_error(uthmani_text=text, ref_ph_text=ref.phonemes, predicted_ph_text=heard, mappings=ref.mappings):
                if not keep(err):
                    continue
                start, end = err.uthmani_pos
                word, letter, key = text[:start].count(" "), letter_at(text, start, end), error_key(err)
                j0, j1 = m[err.ph_pos[0]], m[err.ph_pos[1]]
                conf = min(probs[off + j0: off + j1]) if probs and j1 > j0 else None
                if (word, letter, key) not in items:
                    items[(word, letter, key)] = {"positionIndex": word, "letter": letter, "errorType": key,
                                                  "kind": err.error_type, "speech": err.speech_error_type,
                                                  "expected": err.expected_ph, "heard": err.preditected_ph,
                                                  "tajweed": tajweed_names(err), "conf": conf}
            if len(ayahs) == 1:                            # sifat are only aligned reliably for one ayah at a time
                for ph_idx, letter, key in sifat_errors(ref, out.sifat, heard):
                    ui = uthmani_index_of_phoneme(ref.mappings, ph_idx)
                    word = text[:ui].count(" ")
                    items.setdefault((word, letter, key), {"positionIndex": word, "letter": letter, "errorType": key,
                                                           "kind": "sifat", "speech": "replace", "expected": letter,
                                                           "heard": letter, "tajweed": None, "conf": None})
            per_ayah[(s, a)] = items
        variants.append(per_ayah)
    meta = {"predicted": outs[0].phonemes.text, "reference": whole.phonemes, "inferenceSeconds": round(infer_s, 2),
            "difference": round(lv.distance(whole.phonemes, outs[0].phonemes.text) / max(len(whole.phonemes), 1), 3)}
    return variants, meta


def combine(variants, ayahs, votes=None, min_conf=None, calibration=None, calib_mode=None, skip_kinds=None) -> dict:
    """Keep the mistakes heard in enough versions and, for replaced or added sounds, heard with enough confidence,
    and drop the model's habitual slips on that ayah (calibration)."""
    votes = VOTES if votes is None else votes
    min_conf = MIN_CONF if min_conf is None else min_conf
    calibration = (CALIBRATION if USE_CALIBRATION else {}) if calibration is None else calibration
    calib_mode = CALIB_MODE if calib_mode is None else calib_mode
    skip_kinds = SKIP_KINDS if skip_kinds is None else skip_kinds
    result = {}
    for s, a in ayahs:
        found = {}
        for v in variants:
            for k, item in v[(s, a)].items():
                found.setdefault(k, []).append(item)
        kept, habitual = [], calibration.get(f"{s}:{a}", set())
        habitual_words = {w for w, _ in habitual}
        for k, seen in found.items():
            first = seen[0]
            if len(seen) < votes or first["kind"] in skip_kinds:
                continue
            if (first["positionIndex"], first["errorType"]) in habitual or (
                    calib_mode == "word" and first["positionIndex"] in habitual_words):
                continue
            confs = [x["conf"] for x in seen if x["conf"] is not None]
            if seen[0]["speech"] in ("replace", "insert") and confs and max(confs) < min_conf:
                continue
            item = dict(seen[0]); item["votes"] = len(seen); item["conf"] = max(confs) if confs else None
            kept.append(item)
        result[f"{s}:{a}"] = sorted(kept, key=lambda x: x["positionIndex"])
    return result


def analyze_wave(wave: np.ndarray, ayahs: list[tuple[int, int]]) -> dict:
    """The whole analysis for one recording of the given ayahs (also used by evaluate.py)."""
    variants, meta = raw_mistakes(wave, ayahs)
    if meta["difference"] > MISMATCH:                      # another passage, or only part of it, was recited
        result = {f"{s}:{a}": [] for s, a in ayahs}
        result["_mismatch"] = True
        result["_meta"] = dict(meta, seconds=round(len(wave) / SR, 2), device=DEVICE)
        return result
    result = combine(variants, ayahs)
    result["_meta"] = dict(meta, seconds=round(len(wave) / SR, 2), device=DEVICE, votes=VOTES, minConf=MIN_CONF,
                           calibration=USE_CALIBRATION and bool(CALIBRATION), calibMode=CALIB_MODE,
                           skipKinds=sorted(SKIP_KINDS))
    return result


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=int(os.environ.get("PORT", "8000")))


In [ ]:
%%writefile evaluate.py
"""
Faseeh evaluation: the numbers for the report.

Data is split so the test numbers are honest:
  calibration  Husary, Minshawy, Abdul Basit, Sudais, Shuraym, Ghamdi (calibrate.py) -> habitual slips
  tuning       Hudhaify, Muhammad Ayyoub + even qdat_bench samples -> `sweep` picks the settings
  test         Alafasy, Maher al-Muaiqly + odd qdat_bench samples  -> `test`, the numbers to report

Commands (run in Colab, in this order):
  !python evaluate.py sweep      try every setting on the tuning data, write config.json and results/sweep.csv
  !python evaluate.py test       final numbers on the test data with config.json
  !python evaluate.py own        your own recordings (own_recordings/labels.csv, see below)

own_recordings/labels.csv:
  file,surah,ayah,word,letter
  ali_112_1_ok.webm,112,1,,            correct reading: word empty
  ali_112_1_ha.webm,112,1,3,ح         mistake on word 3 (counting from 0), letter ح

Metrics
  expert pass rate   share of correctly recited ayahs with no mistake reported (false alarms = 1 - this)
  precision          of the mistakes reported, how many were real
  recall             of the real mistakes, how many were reported
  F1                 balance of the two
  PER                phoneme error rate of the model's transcription
"""
import csv
import itertools
import json
import os
import sys
import urllib.request
from collections import Counter

import Levenshtein as lv
import numpy as np

import faseeh_server as fs
from quran_transcript import Aya, explain_error, quran_phonetizer

OUT = os.environ.get("FASEEH_RESULTS", "results")
os.makedirs(OUT, exist_ok=True)
APP_AYAHS = [(s, a) for s, n in {78: 40, 80: 42, 103: 3, 108: 3, 112: 4, 113: 5, 114: 6}.items() for a in range(1, n + 1)]
TUNE_RECITERS = ["Hudhaify_128kbps", "Muhammad_Ayyoub_128kbps"]
TEST_RECITERS = ["Alafasy_128kbps", "MaherAlMuaiqly128kbps"]
BENCH = (5, 109)
GRID = {"votes": [2, 3], "min_conf": [0.0, 0.5, 0.7, 0.8, 0.9, 0.95], "calibration": [False, True],
        "calib_mode": ["exact", "word"], "skip_kinds": [(), ("tashkeel",)]}
TARGET_PASS = float(os.environ.get("FASEEH_TARGET_PASS", "1.0"))
MIN_RECALL = float(os.environ.get("FASEEH_MIN_RECALL", "0.75"))


def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return round(p, 4), round(r, 4), round(2 * p * r / (p + r), 4) if p + r else 0.0


def write_csv(name, rows):
    if rows:
        with open(os.path.join(OUT, name), "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)


def save_summary(key, value):
    path = os.path.join(OUT, "summary.json")
    s = json.load(open(path)) if os.path.exists(path) else {}
    s[key] = value
    json.dump(s, open(path, "w"), ensure_ascii=False, indent=2)
    print(json.dumps(value, ensure_ascii=False, indent=2))


# ---------------------------------------------------------------- data
def expert_raw(reciters):
    """Model output (all variants, before any filtering) for every app ayah read by each expert reciter."""
    if isinstance(reciters, str):
        reciters = [reciters]
    raw = []
    for reciter, (s, a) in itertools.product(reciters, APP_AYAHS):
        try:
            audio = urllib.request.urlopen(f"https://everyayah.com/data/{reciter}/{s:03d}{a:03d}.mp3", timeout=60).read()
            wave = fs.to_wave(audio)
        except Exception as e:
            print("skip", s, a, e); continue
        if len(wave) / fs.SR > fs.MAX_SECONDS:
            continue
        raw.append(((s, a), fs.raw_mistakes(wave, [(s, a)])[0], reciter))
        print(f"{reciter} {s}:{a}", end="      \r")
    print()
    return raw


def truth_mistakes(text, ref, said):
    out = set()
    for e in explain_error(uthmani_text=text, ref_ph_text=ref.phonemes, predicted_ph_text=said, mappings=ref.mappings):
        if fs.keep(e):
            out.add((text[:e.uthmani_pos[0]].count(" "), e.error_type))
    return out


def bench_raw(parity):
    """qdat_bench samples with index % 2 == parity: true mistakes, model output and PER counts."""
    from datasets import load_dataset, Audio
    ds = load_dataset("obadx/qdat_bench", split="train").cast_column("audio", Audio(sampling_rate=fs.SR))
    text = Aya(sura_idx=BENCH[0], aya_idx=BENCH[1]).get().uthmani
    ref = quran_phonetizer(text, fs.MOSHAF, remove_spaces=True, sura_idx=BENCH[0])
    raw = []
    for i, item in enumerate(ds):
        if i % 2 != parity:
            continue
        wave = np.asarray(item["audio"]["array"], dtype=np.float32)
        variants, meta = fs.raw_mistakes(wave, [BENCH])
        truth = item["phonetic_transcript"]
        raw.append({"id": item["id"], "gender": item["gender"], "truth": truth_mistakes(text, ref, truth),
                    "variants": variants, "edits": lv.distance(meta["predicted"], truth), "chars": len(truth)})
        print(f"qdat_bench {i + 1}/{len(ds)}", end="      \r")
    print()
    return raw


# ---------------------------------------------------------------- scoring one setting
def score_experts(raw, **cfg):
    rows = []
    for (s, a), variants, reciter in raw:
        found = fs.combine(variants, [(s, a)], **cfg)[f"{s}:{a}"]
        rows.append({"reciter": reciter, "surah": s, "ayah": a, "reported": len(found),
                     "types": "; ".join(f'{e["positionIndex"]}:{e["errorType"]}' for e in found)})
    clean = sum(r["reported"] == 0 for r in rows)
    return {"ayahs": len(rows), "passed_clean": clean, "pass_rate": round(clean / max(len(rows), 1), 4),
            "false_alarms": sum(r["reported"] for r in rows)}, rows


def score_bench(raw, **cfg):
    tot, by_kind, rows = Counter(), {}, []
    key = f"{BENCH[0]}:{BENCH[1]}"
    for r in raw:
        found = fs.combine(r["variants"], [BENCH], **cfg)[key]
        det = {(e["positionIndex"], e["kind"]) for e in found if e["kind"] != "sifat"}   # sifat are not annotated
        t = r["truth"]
        tot.update(tp=len(t & det), fp=len(det - t), fn=len(t - det))
        for kind in ("normal", "tashkeel", "tajweed"):
            tk, dk = {x for x in t if x[1] == kind}, {x for x in det if x[1] == kind}
            by_kind.setdefault(kind, Counter()).update(tp=len(tk & dk), fp=len(dk - tk), fn=len(tk - dk))
        rows.append({"id": r["id"], "gender": r["gender"], "true": len(t), "reported": len(det),
                     "tp": len(t & det), "fp": len(det - t), "fn": len(t - det)})
    p, rc, f = prf(tot["tp"], tot["fp"], tot["fn"])
    per = round(sum(r["edits"] for r in raw) / max(sum(r["chars"] for r in raw), 1), 4)
    kinds = {k: dict(zip(("precision", "recall", "f1"), prf(c["tp"], c["fp"], c["fn"])), true=c["tp"] + c["fn"])
             for k, c in by_kind.items()}
    return {"samples": len(raw), "PER": per, "precision": p, "recall": rc, "f1": f, "by_type": kinds}, rows


# ---------------------------------------------------------------- commands
def sweep():
    ex, bn = expert_raw(TUNE_RECITERS), bench_raw(parity=0)
    rows = []
    for votes, min_conf, cal, mode, skip in itertools.product(*GRID.values()):
        if not cal and mode == "word":
            continue
        cfg = dict(votes=votes, min_conf=min_conf, calibration=fs.CALIBRATION if cal else {}, calib_mode=mode,
                   skip_kinds=set(skip))
        e, _ = score_experts(ex, **cfg)
        b, _ = score_bench(bn, **cfg)
        rows.append({"votes": votes, "min_conf": min_conf, "calibration": cal, "calib_mode": mode,
                     "skip_kinds": "+".join(skip), "expert_pass": e["pass_rate"], "false_alarms": e["false_alarms"],
                     "precision": b["precision"], "recall": b["recall"], "f1": b["f1"]})
    write_csv("sweep.csv", rows)
    # Fewest false alarms first; among equals the best F1. A setting must still catch most real mistakes.
    usable = [r for r in rows if r["recall"] >= MIN_RECALL] or rows
    ok = [r for r in usable if r["expert_pass"] >= TARGET_PASS]
    best = max(usable, key=lambda r: (r["expert_pass"], r["f1"]))
    cfg = {"votes": best["votes"], "min_conf": best["min_conf"], "calibration": best["calibration"],
           "calib_mode": best["calib_mode"], "skip_kinds": [k for k in best["skip_kinds"].split("+") if k]}
    json.dump(cfg, open(os.path.join(fs.HERE, "config.json"), "w"))
    print("\nsetting  pass  F1")
    for r in sorted(rows, key=lambda r: (-r["expert_pass"], -r["f1"]))[:12]:
        print(f'votes {r["votes"]}  conf {r["min_conf"]:.2f}  calib {str(r["calibration"]):5} {r["calib_mode"]:5}  skip {r["skip_kinds"] or "-":8}'
              f'   pass {r["expert_pass"]:.1%}  false alarms {r["false_alarms"]:3d}  F1 {r["f1"]:.3f}  recall {r["recall"]:.3f}')
    save_summary("sweep_choice", dict(cfg, tuning_expert_pass=best["expert_pass"], tuning_f1=best["f1"],
                                      tuning_recall=best["recall"], target_pass=TARGET_PASS,
                                      met_target=bool(ok), min_recall=MIN_RECALL))


def test():
    cfg = dict(votes=fs.VOTES, min_conf=fs.MIN_CONF, calibration=fs.CALIBRATION if fs.USE_CALIBRATION else {},
               calib_mode=fs.CALIB_MODE, skip_kinds=fs.SKIP_KINDS)
    ex, bn = expert_raw(TEST_RECITERS), bench_raw(parity=1)
    base = dict(votes=1, min_conf=0.0, calibration={}, skip_kinds=set())   # no false-alarm filtering, for comparison
    e0, _ = score_experts(ex, **base); b0, _ = score_bench(bn, **base)
    e1, erows = score_experts(ex, **cfg); b1, brows = score_bench(bn, **cfg)
    write_csv("test_experts.csv", erows); write_csv("test_benchmark.csv", brows)
    per_reciter = {r: dict(zip(("ayahs", "passed_clean"), (sum(x["reciter"] == r for x in erows),
                                                          sum(x["reciter"] == r and x["reported"] == 0 for x in erows))))
                   for r in TEST_RECITERS}
    save_summary("test", {"settings": {"votes": fs.VOTES, "min_conf": fs.MIN_CONF, "calibration": fs.USE_CALIBRATION,
                                       "calib_mode": fs.CALIB_MODE, "skip_kinds": sorted(fs.SKIP_KINDS)},
                          "per_reciter": per_reciter,
                          "without_filtering": {"experts": e0, "benchmark": b0},
                          "final": {"experts": e1, "benchmark": b1}})


def own(folder="own_recordings"):
    labels = os.path.join(folder, "labels.csv")
    if not os.path.exists(labels):
        return save_summary("own", {"note": f"no {labels} yet"})
    expected = {}
    for r in csv.DictReader(open(labels, encoding="utf-8")):
        k = (r["file"], int(r["surah"]), int(r["ayah"]))
        expected.setdefault(k, set())
        if (r.get("word") or "").strip():
            expected[k].add(int(r["word"]))
    rows, tot = [], Counter()
    for (file, s, a), words in expected.items():
        res = fs.analyze_wave(fs.to_wave(open(os.path.join(folder, file), "rb").read()), [(s, a)])[f"{s}:{a}"]
        found = {e["positionIndex"] for e in res}
        tot.update(tp=len(words & found), fp=len(found - words), fn=len(words - found))
        rows.append({"file": file, "surah": s, "ayah": a, "expected": sorted(words), "found": sorted(found),
                     "details": "; ".join(f'{e["positionIndex"]}:{e["letter"]}:{e["errorType"]}' for e in res)})
    write_csv("own.csv", rows)
    p, r, f = prf(tot["tp"], tot["fp"], tot["fn"])
    clean = [x for x in rows if not x["expected"]]
    save_summary("own", {"recordings": len(rows), "precision": p, "recall": r, "f1": f,
                         "correct_readings": len(clean), "correct_passed": sum(not x["found"] for x in clean)})


if __name__ == "__main__":
    {"sweep": sweep, "test": test, "own": own}[sys.argv[1] if len(sys.argv) > 1 else "test"]()


In [ ]:
%%writefile calibrate.py
"""
Calibration: find the model's habitual slips.

Expert reciters read every ayah correctly, so anything the model "hears" as a mistake in their recitation is a
slip of the model at that spot (for example it tends to hear a ث as س in one particular word). Those spots are
saved in calibration.json and the server stops reporting that same mistake type on that same word.

Reciters used here are NOT used for tuning or testing (see evaluate.py), so the test numbers stay honest.

Run once:  !python calibrate.py      (about 10 minutes on a T4)
"""
import json
import os
import urllib.request

import faseeh_server as fs

RECITERS = ["Husary_128kbps", "Minshawy_Murattal_128kbps", "Abdul_Basit_Murattal_192kbps",
            "Abdurrahmaan_As-Sudais_192kbps", "Saood_ash-Shuraym_128kbps", "Ghamadi_40kbps"]
# For the website after the evaluation is done: `!python calibrate.py all` also calibrates on the tuning and test
# reciters (more reciters = more habitual slips known). Only do this AFTER `evaluate.py test`, and report the
# numbers from the evaluation, which never saw these reciters during calibration.
import sys
if len(sys.argv) > 1 and sys.argv[1] == "all":
    RECITERS += ["Hudhaify_128kbps", "Muhammad_Ayyoub_128kbps", "Alafasy_128kbps", "MaherAlMuaiqly128kbps"]
AYAHS = [(s, a) for s, n in {78: 40, 80: 42, 103: 3, 108: 3, 112: 4, 113: 5, 114: 6}.items() for a in range(1, n + 1)]
AYAHS.append((5, 109))          # the benchmark ayah, so the benchmark measures what calibration costs


def everyayah(reciter, s, a):
    url = f"https://everyayah.com/data/{reciter}/{s:03d}{a:03d}.mp3"
    return fs.to_wave(urllib.request.urlopen(url, timeout=60).read())


def main():
    slips = {}
    for reciter in RECITERS:
        done = 0
        for s, a in AYAHS:
            try:
                wave = everyayah(reciter, s, a)
            except Exception as e:
                print("skip", reciter, s, a, e); continue
            variants, _ = fs.raw_mistakes(wave, [(s, a)])
            # the most permissive setting (one vote, any confidence): catch every slip the model can make here
            found = fs.combine(variants, [(s, a)], votes=1, min_conf=0.0, calibration={})[f"{s}:{a}"]
            for e in found:
                slips.setdefault(f"{s}:{a}", {}).setdefault(f'{e["positionIndex"]}|{e["errorType"]}', []).append(reciter)
            done += 1
            print(f"{reciter}  {s}:{a}  slips {len(found)}", end="     \r")
        print(f"\n{reciter}: {done} ayahs")
    out = {k: [[int(w.split("|")[0]), w.split("|")[1]] for w in v] for k, v in slips.items()}
    json.dump(out, open(os.path.join(fs.HERE, "calibration.json"), "w"), ensure_ascii=False, indent=1)
    json.dump(slips, open(os.path.join(fs.HERE, "calibration_detail.json"), "w"), ensure_ascii=False, indent=1)
    print("habitual slips:", sum(len(v) for v in out.values()), "in", len(out), "of", len(AYAHS), "ayahs")


if __name__ == "__main__":
    main()


## 2. Settings, calibration and start the server

**Already ran the evaluation before?** Run the first cell below and choose your `evaluation_results.zip`: it restores `config.json` and `calibration.json`.

**First time ever?** Skip it, run `calibrate.py`, then section 4 (`sweep`, `test`) before using the site.

`calibrate.py all` (about 40 minutes) calibrates on all 10 reciters for the website. Run it after the evaluation numbers are saved.

In [ ]:
# Restore the settings from evaluation_results.zip
from google.colab import files
import zipfile, io
for name, data in files.upload().items():
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        for f in ("config.json", "calibration.json"):
            if f in z.namelist():
                open(f, "wb").write(z.read(f)); print("restored", f)

In [ ]:
!python calibrate.py all      # for the website: all 10 reciters (about 40 minutes)

In [ ]:
import subprocess, time, urllib.request
server = subprocess.Popen(["python", "faseeh_server.py"], stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
for i in range(120):
    try:
        print(urllib.request.urlopen("http://localhost:8000/health", timeout=3).read().decode()); break
    except Exception:
        time.sleep(5)
else:
    print(open("server.log").read()[-3000:])

## 3. Public link for the website
The link changes every time Colab restarts; open the printed website link again after a restart.

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import re, subprocess, time
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0); break
print("Server:", url)
SITE = "https://YOUR-SITE"   # <- your Faseeh website address
print("Open the website with:", f"{SITE}/?server={url}")

In [ ]:
# Quick check: Alafasy reading 112:1 should give no (or almost no) mistakes
import requests, json
audio = requests.get("https://everyayah.com/data/Alafasy_128kbps/112001.mp3").content
r = requests.post("http://localhost:8000/analyze", data=audio, headers={"X-Ayahs": "112:1"})
print(json.dumps(r.json(), ensure_ascii=False, indent=1))

## 4. Evaluation

The data is split so the reported numbers are honest:

| Part | Data | Used for |
|---|---|---|
| calibration | Husary, Minshawy, Abdul Basit, Sudais, Shuraym, Ghamdi | the model's habitual slips (`calibrate.py`) |
| tuning | Hudhaify, Muhammad Ayyoub + even qdat_bench samples | `sweep` chooses the false-alarm settings (`config.json`) |
| test | Alafasy, Maher al-Muaiqly + odd qdat_bench samples | `test`: the numbers for the report |

After `sweep`, restart the server (cell below) so it uses `config.json`.

Your own recordings go in `own_recordings/` with `labels.csv`:
```
file,surah,ayah,word,letter
ali_112_1_ok.webm,112,1,,
ali_112_1_ha.webm,112,1,3,ح
```
(word = index of the wrong word, counting from 0; empty for a correct reading)

In [ ]:
!python evaluate.py sweep

In [ ]:
# restart the server with the chosen settings
server.terminate(); server.wait()
server = subprocess.Popen(["python", "faseeh_server.py"], stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
for i in range(120):
    try:
        print(urllib.request.urlopen("http://localhost:8000/health", timeout=3).read().decode()); break
    except Exception:
        time.sleep(5)
print(open("server.log").read()[-600:])

In [ ]:
!python evaluate.py test

In [ ]:
import os
os.makedirs("own_recordings", exist_ok=True)
from google.colab import files
up = files.upload()            # choose labels.csv and the audio files
for name, data in up.items():
    open(os.path.join("own_recordings", name), "wb").write(data)
!python evaluate.py own

In [ ]:
# Charts for the report
import json, matplotlib.pyplot as plt
T = json.load(open("results/summary.json"))["test"]
before, after = T["without_filtering"], T["final"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(["without filtering", "Faseeh"], [before["experts"]["pass_rate"], after["experts"]["pass_rate"]], color=["#c0563b", "#0d6b62"])
axes[0].set_ylim(0, 1); axes[0].set_title(f"Correct recitations passed (Alafasy, {after['experts']['ayahs']} ayahs)")
m = ["precision", "recall", "f1"]
x = range(len(m))
axes[1].bar([i - 0.2 for i in x], [before["benchmark"][k] for k in m], 0.4, label="without filtering", color="#c0563b")
axes[1].bar([i + 0.2 for i in x], [after["benchmark"][k] for k in m], 0.4, label="Faseeh", color="#0d6b62")
axes[1].set_xticks(list(x), m); axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].set_title("Learner mistakes (qdat_bench test half)")
plt.tight_layout(); plt.savefig("results/charts.png", dpi=200); plt.show()
print(json.dumps(T, ensure_ascii=False, indent=1))

In [ ]:
!zip -qr results.zip results && echo "Download results.zip from the Files panel"

## Optional: the reciter's recordings for the website (one time)
Downloads the surah recordings the word timings were made for, as `faseeh_audio.zip`. Unzip it and upload the `audio` folder to the root of the GitHub repository: the website then highlights every word exactly, even when this server is off.


In [ ]:
import urllib.request, zipfile
SURAHS = [78, 80, 103, 108, 112, 113, 114]          # the surahs in data/quran.json
with zipfile.ZipFile("faseeh_audio.zip", "w") as z:
    for s in SURAHS:
        req = urllib.request.Request(f"https://download.quranicaudio.com/qdc/mishari_al_afasy/murattal/{s}.mp3", headers={"User-Agent": "Faseeh"})
        data = urllib.request.urlopen(req, timeout=120).read()
        z.writestr(f"audio/{s}.mp3", data)
        print(s, len(data) // 1024, "KB")
from google.colab import files; files.download("faseeh_audio.zip")
